### Necessary imports

In [2]:
import pandas as pd
import sqlite3
import json
from pathlib import Path


### Create db

In [11]:
db_path = "../01_data/wti_thesis.db"
conn = sqlite3.connect(db_path)
print(f"Database created at {db_path}")

Database created at ../01_data/wti_thesis.db


### DB tables creation

In [ ]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS articles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    datetime TEXT,
    domain TEXT,
    url TEXT,
    body TEXT,
    body_valid INTEGER,
    assignment_gap REAL
);

CREATE TABLE IF NOT EXISTS liquidity (
    article_id INTEGER PRIMARY KEY,
    datetime_hour TEXT,
    close REAL,
    volume REAL,
    log_volume REAL,
    price_range REAL,
    log_return REAL,
    amihud REAL,
    FOREIGN KEY (article_id) REFERENCES articles(id)
);

CREATE TABLE IF NOT EXISTS llm_features (
    article_id INTEGER PRIMARY KEY,
    sentiment_score REAL,
    magnitude REAL,
    event_type TEXT,
    entities TEXT,
    certainty REAL,
    price_direction TEXT,
    time_horizon TEXT,
    FOREIGN KEY (article_id) REFERENCES articles(id)
);

CREATE TABLE IF NOT EXISTS market_context (
    datetime_hour TEXT PRIMARY KEY,
    log_volume REAL,
    price_range REAL,
    log_return REAL,
    amihud REAL,
    dxy REAL,
    vix REAL,
    cushing_inventory REAL
);

CREATE TABLE IF NOT EXISTS opec_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    datetime TEXT,
    event_type TEXT,
    decision TEXT,
    production_change_mbpd REAL
);
                   
CREATE TABLE IF NOT EXISTS eia_events (
    date TEXT PRIMARY KEY,
    inventory_mbbl REAL,
    inventory_change REAL,
    news_direction TEXT,
    datetime_et TEXT
);
""")
conn.commit()
print("Tables created successfully")

Tables created successfully


### Migrate data from CSVs

Migrate aligned articles + liquidity

In [5]:

df = pd.read_csv("../01_data/processed/gdelt_wti_aligned.csv", parse_dates=['datetime'])

# Add explicit ID column
df = df.reset_index(drop=True)
df['id'] = df.index + 1

# Articles table
articles_cols = ['id', 'title', 'datetime', 'domain', 'url', 'body', 'body_valid', 'assignment_gap']
df_articles = df[articles_cols].copy()
df_articles['datetime'] = df_articles['datetime'].astype(str)
df_articles.to_sql('articles', conn, if_exists='replace', index=False)

# Liquidity table
liquidity_cols = ['id', 'datetime_hour', 'Close', 'Volume', 'log_volume', 'price_range', 'log_return', 'amihud']
df_liquidity = df[liquidity_cols].copy()
df_liquidity.columns = ['article_id', 'datetime_hour', 'close', 'volume',
                         'log_volume', 'price_range', 'log_return', 'amihud']
df_liquidity['datetime_hour'] = df_liquidity['datetime_hour'].astype(str)
df_liquidity.to_sql('liquidity', conn, if_exists='replace', index=False)

print(f"Articles migrated: {len(df_articles)}")
print(f"Liquidity records migrated: {len(df_liquidity)}")

Articles migrated: 13691
Liquidity records migrated: 13691


Migrate price series to market_context

In [ ]:
df_price = pd.read_csv("../01_data/raw/price/wti_hourly_raw.csv", 
                        index_col='Datetime', parse_dates=True)
df_price.index = pd.to_datetime(df_price.index, utc=True)

df_market = df_price[['log_volume', 'price_range', 'log_return', 'amihud']].copy()
df_market.index.name = 'datetime_hour'
df_market = df_market.reset_index()
df_market['datetime_hour'] = df_market['datetime_hour'].astype(str)

# Add empty columns for future features
df_market['dxy'] = None
df_market['vix'] = None
df_market['cushing_inventory'] = None

df_market.to_sql('market_context', conn, if_exists='replace', index=False)
print(f"Market context records: {len(df_market)}")

Market context records: 11219


Verify migration

In [ ]:
print("=== Database Summary ===")
for table in ['articles', 'liquidity', 'market_context', 'llm_features', 'opec_events']:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn)['n'][0]
    print(f"{table}: {count} records")

# Test a query
sample = pd.read_sql("""
    SELECT a.title, a.datetime, l.log_volume, l.price_range
    FROM articles a
    JOIN liquidity l ON a.id = l.article_id
    LIMIT 5
""", conn)
print(f"\nSample join query:")
print(sample)

conn.close()
print("\nDatabase ready.")

=== Database Summary ===
articles: 13691 records
liquidity: 13691 records
market_context: 11219 records
llm_features: 0 records
opec_events: 0 records

Sample join query:
                                               title  \
0  Oil prices fall amid Chinese demand concerns ,...   
1  Asian crude buyers unlikely to face supply squ...   
2                     Oil Edges Up In Cautious Trade   
3  Futures Drop As Nvidia Extends Slide , Japan S...   
4  Weekly Commentary : Q4 2023 Z . 1 : Bubble Con...   

                    datetime  log_volume  price_range  
0  2024-03-11 10:30:00+00:00    9.357121     0.330002  
1  2024-03-11 13:00:00+00:00   11.046324     1.180000  
2  2024-03-11 13:15:00+00:00   10.543366     0.879997  
3  2024-03-11 14:15:00+00:00   10.372459     0.909996  
4  2024-03-11 14:15:00+00:00   10.372459     0.909996  

Database ready.


### Add EIA features to market_context

In [10]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_eia = pd.read_sql("SELECT date, inventory_change, datetime_et FROM eia_events", conn)

# Convert to UTC explicitly
df_eia['datetime_et'] = pd.to_datetime(df_eia['datetime_et'], utc=True)
df_eia = df_eia.sort_values('datetime_et').reset_index(drop=True)

df_market = pd.read_sql("SELECT * FROM market_context", conn)
df_market['datetime_hour'] = pd.to_datetime(df_market['datetime_hour'], utc=True)
df_market['eia_surprise'] = 0.0
df_market['is_eia_release'] = 0

for idx, market_row in df_market.iterrows():
    hour = market_row['datetime_hour']
    past_eia = df_eia[df_eia['datetime_et'] <= hour]
    if len(past_eia) > 0:
        latest = past_eia.iloc[-1]
        df_market.at[idx, 'eia_surprise'] = latest['inventory_change']
        if latest['datetime_et'].floor('h') == hour:
            df_market.at[idx, 'is_eia_release'] = 1

df_market['datetime_hour'] = df_market['datetime_hour'].astype(str)
df_market.to_sql('market_context', conn, if_exists='replace', index=False)
print("Done")
conn.close()

Done
